## Course data paths

Run the setup cell below before the lesson. Bundled inputs are resolved from the course repository, so paths also work after moving the notebook into `code/`. Generated practice files still use the notebook working directory.


In [ ]:
from pathlib import Path

# Find the cloned course repository from the notebook or repository folder.
COURSE_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "docs" / "course-catalog.json").is_file()),
    None,
)
if COURSE_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the cloned course repository.")


In [ ]:
# ============ HOUR 1: Introduction to Pandas ============

# T01: Import libraries and print versions
import pandas as pd
import numpy as np

# T02: Load the heart disease CSV into DataFrame
df = pd.read_csv(COURSE_ROOT / "modules/03-pandas/datasets/heart_disease.csv")
print("DataFrame shape:", df.shape)

# T03: Create a small Series of 5 patient ages
s = pd.Series([45, 52, 38, 61, 29], index=['P1', 'P2', 'P3', 'P4', 'P5'], name='Age')
print("Series created:\n", s)

# T04: Create a DataFrame manually from a dictionary
df_manual = pd.DataFrame({
    'Age': [45, 52, 38, 61, 29],
    'BMI': [28.5, 32.1, 24.3, 29.8, 22.5],
    'Smoking': ['Yes', 'No', 'No', 'Yes', 'No'],
    'Heart Disease Status': ['Yes', 'No', 'No', 'Yes', 'No']
})
print("Manual DataFrame created:\n", df_manual)

# T05: Print column names and row index
print("Column names:", df.columns.tolist())
print("Row index:", df.index.tolist())


# ============ HOUR 2: Series, DataFrame & Selection/Indexing ============

# T06: Select only the 'Age' column
col = df['Age']
print("Type of 'Age' column:", type(col))

# T07: Select multiple columns
sub = df[['Age', 'BMI', 'Heart Disease Status']]
print("Type of selected columns:", type(sub))

# T08: Use loc[] to select rows 0 to 9 and columns 'Age' through 'BMI'
result_t08 = df.loc[0:9, 'Age':'BMI']
print("T08 result shape:", result_t08.shape)

# T09: Use loc[] to select first 5 rows and last 3 columns
result_t09 = df.loc[0:4, -3:]
print("T09 result shape:", result_t09.shape)

# T10: Filter patients where BMI > 30
obese = df[df['BMI'] > 30]
print("Number of patients with BMI > 30:", len(obese))

# T11: Filter patients who Smoke AND have Diabetes
smoking_diabetes = df[(df['Smoking'] == 'Yes') & (df['Diabetes'] == 'Yes')][['Age', 'BMI', 'Heart Disease Status']]
print("Smoking & Diabetes patients count:", len(smoking_diabetes))

# T12: Filter patients where Stress Level is 'High' OR Sleep Hours < 5
high_stress_low_sleep = df[(df['Stress Level'] == 'High') | (df['Sleep Hours'] < 5)]
print("High stress OR low sleep patients count:", len(high_stress_low_sleep))

# T13: Use isin() for Alcohol Consumption
alcohol_filter = df[df['Alcohol Consumption'].isin(['Heavy', 'Moderate'])]
print("Alcohol consumption filter result count:", len(alcohol_filter))

# T14: Use query() for Age > 50 with Heart Disease
query_result = df.query("Age > 50 and `Heart Disease Status` == 'Yes'")
print("Query result count:", len(query_result))

# T15: Set 'Age' as index and access row with label 35
df2 = df.set_index('Age')
row_35 = df2.loc[35] if 35 in df2.index else "Age 35 not found"
print("Row with Age 35:", row_35)
df = df.reset_index(drop=True)  # Restore original


# ============ HOUR 3: Operations on DataFrames ============

# ============ HOUR 4: Missing Data and Its Handling ============

# T28: Identify missing values
print("Missing values per column:\n", df.isna().sum())

# T29: Calculate percentage of missing values
miss_pct = df.isna().sum() / len(df) * 100
print("Columns with missing values (>0%):\n", miss_pct[miss_pct > 0])

# T30: Replace 0s with NaN in specified columns
cols_with_zeros = ['BMI', 'Triglyceride Level', 'Fasting Blood Sugar', 'CRP Level', 'Homocysteine Level']
available_cols = [col for col in cols_with_zeros if col in df.columns]
df[available_cols] = df[available_cols].replace(0, np.nan)
print(f"Replaced zeros with NaN in: {available_cols}")

# T31: Check rows lost if dropping all NaN rows
rows_to_lose = len(df) - len(df.dropna())
print(f"Would lose: {rows_to_lose} rows if dropping all NaN rows")

# T32: Drop rows only where BMI is missing
df_clean = df.dropna(subset=['BMI'])
print(f"Original rows: {len(df)}, After dropping missing BMI: {len(df_clean)}")

# T33: Fill missing Triglyceride Level with mean
before_count = df['Triglyceride Level'].isna().sum()
df['Triglyceride Level'].fillna(df['Triglyceride Level'].mean(), inplace=True)
after_count = df['Triglyceride Level'].isna().sum()
print(f"Triglyceride Level - Before: {before_count} NaN, After: {after_count} NaN")

# T34: Fill missing CRP Level with median (better for skewed data)
before_count = df['CRP Level'].isna().sum()
df['CRP Level'].fillna(df['CRP Level'].median(), inplace=True)
after_count = df['CRP Level'].isna().sum()
print(f"CRP Level (median fill) - Before: {before_count} NaN, After: {after_count} NaN")
print("Median is better for CRP because it's robust to outliers in skewed distributions")

# T35: Fill missing Fasting Blood Sugar using group median by Diabetes
if 'Diabetes' in df.columns and 'Fasting Blood Sugar' in df.columns:
    grp_median = df.groupby('Diabetes')['Fasting Blood Sugar'].transform('median')
    before_count = df['Fasting Blood Sugar'].isna().sum()
    df['Fasting Blood Sugar'] = df['Fasting Blood Sugar'].fillna(grp_median)
    after_count = df['Fasting Blood Sugar'].isna().sum()
    print(f"Fasting Blood Sugar (group median) - Before: {before_count} NaN, After: {after_count} NaN")

# T36: Use SimpleImputer with median strategy
from sklearn.impute import SimpleImputer
impute_cols = ['BMI', 'Homocysteine Level', 'CRP Level']
available_impute_cols = [col for col in impute_cols if col in df.columns]

if available_impute_cols:
    imp = SimpleImputer(strategy='median')
    imputed_values = imp.fit_transform(df[available_impute_cols])
    df[available_impute_cols] = imputed_values
    print(f"SimpleImputer applied to: {available_impute_cols}")

# T37: Complete pipeline - load fresh, replace zeros, drop duplicates, impute, save
print("\n" + "="*50)
print("T37: Running complete pipeline")
print("="*50)

# Load fresh copy
df_final = pd.read_csv(COURSE_ROOT / "modules/03-pandas/datasets/heart_disease.csv")

# Replace zeros with NaN in specified columns
for col in cols_with_zeros:
    if col in df_final.columns:
        df_final[col] = df_final[col].replace(0, np.nan)

# Drop duplicates
df_final.drop_duplicates(inplace=True)

# Impute numeric columns with median
numeric_cols = df_final.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_final[col].isna().any():
        df_final[col].fillna(df_final[col].median(), inplace=True)

# Reset index
df_final.reset_index(drop=True, inplace=True)

# Save to CSV
df_final.to_csv('heart_clean.csv', index=False)
print(f"Clean dataset saved to 'heart_clean.csv'")
print(f"Final shape: {df_final.shape}")
print("Missing values after pipeline:\n", df_final.isna().sum().sum(), "total missing values")

In [ ]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

In [ ]:
# T02: Load the heart disease CSV into DataFrame
df = pd.read_csv(COURSE_ROOT / "modules/03-pandas/datasets/heart_disease.csv")
print("DataFrame shape:", df.shape)

In [ ]:
s = pd.Series([45, 52, 38, 61, 29], index=['P1', 'P2', 'P3', 'P4', 'P5'], name='Age')
print("Series created:\n", s)

In [ ]:
# T04: Create a DataFrame manually from a dictionary
df_manual = pd.DataFrame({
    'Age': [45, 52, 38, 61, 29],
    'BMI': [28.5, 32.1, 24.3, 29.8, 22.5],
    'Smoking': ['Yes', 'No', 'No', 'Yes', 'No'],
    'Heart Disease Status': ['Yes', 'No', 'No', 'Yes', 'No']
})
print("Manual DataFrame created:\n", df_manual)

In [ ]:
# T05: Print column names and row index
print("Column names:", df.columns.tolist())
print("Row index:", df.index.tolist())

In [ ]:

# T06: Select only the 'Age' column
col = df['Age']
print("Type of 'Age' column:", type(col))

# T07: Select multiple columns









In [ ]:
sub = df[['Age', 'BMI', 'Heart Disease Status']]
print("Type of selected columns:", type(sub))


In [ ]:
# T08: Use loc[] to select rows 0 to 9 and columns 'Age' through 'BMI'
result_t08 = df.loc[0:9, 'Age':'BMI']
print("T08 result shape:", result_t08.shape)


In [ ]:

result_t09 = df.iloc[0:5, -3:]
print("T09 result shape:", result_t09.shape)


In [ ]:
# T10: Filter patients where BMI > 30
obese = df[df['BMI'] > 30]
print("Number of patients with BMI > 30:", len(obese))

In [ ]:

# T11: Filter patients who Smoke AND have Diabetes
smoking_diabetes = df[(df['Smoking'] == 'Yes') & (df['Diabetes'] == 'Yes')][['Age', 'BMI', 'Heart Disease Status']]
print("Smoking & Diabetes patients count:", len(smoking_diabetes))


In [ ]:

# T12: Filter patients where Stress Level is 'High' OR Sleep Hours < 5
high_stress_low_sleep = df[(df['Stress Level'] == 'High') | (df['Sleep Hours'] < 5)]
print("High stress OR low sleep patients count:", len(high_stress_low_sleep))

In [ ]:

# T13: Use isin() for Alcohol Consumption
alcohol_filter = df[df['Alcohol Consumption'].isin(['Heavy', 'Moderate'])]
print("Alcohol consumption filter result count:", len(alcohol_filter))


In [ ]:
# T14: Use query() for Age > 50 with Heart Disease
query_result = df.query("Age > 50 and `Heart Disease Status` == 'Yes'")
print("Query result count:", len(query_result))

In [ ]:

# T15: Set 'Age' as index and access row with label 35
df2 = df.set_index('Age')
row_35 = df2.loc[35] if 35 in df2.index else "Age 35 not found"
print("Row with Age 35:", row_35)
df = df.reset_index(drop=True)  # Restore original


In [ ]:

# T16: Print first 7 and last 4 rows
print("First 7 rows:\n", df.head(7))
print("Last 4 rows:\n", df.tail(4))




# T20: Create BMI_Category using apply()


# T21: Create Risk_Score using apply(axis=1)


# T22: Use map() to convert Target Yes/No to 1/0

# T25: Drop columns and confirm shape




In [ ]:
# T17: Print all unique values
print("Unique Exercise Habits:", df['Exercise Habits'].unique())
print("Unique Stress Level:", df['Stress Level'].unique())

#

In [ ]:
 # T18: Value counts on Heart Disease Status

# T19: Rename columns
df.rename(columns={'Heart Disease Status': 'Target', 'Blood Pressure': 'BP'}, inplace=True)
print("Renamed columns:", df.columns.tolist())

In [ ]:
def bmi_category(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'


In [ ]:


df['Risk_Score'] = df.apply(calculate_risk, axis=1)
print("Risk_Score created. Sample:\n", df[['Age', 'BMI', 'Smoking', 'Diabetes', 'Risk_Score']].head())

In [ ]:
df['BMI_Category'] = df['BMI'].apply(bmi_category)
print("BMI_Category created:\n", df[['BMI', 'BMI_Category']].head())


In [ ]:
df['Target'] = df['Target'].map({'Yes': 1, 'No': 0})
print("Target converted to numeric. Unique values:", df['Target'].unique())


In [ ]:

# T23: Sort by Age descending, then BMI ascending
sorted_df = df.sort_values(by=['Age', 'BMI'], ascending=[False, True])
print("Top 10 sorted:\n", sorted_df[['Age', 'BMI']].head(10))

# T24: Find highest Cholesterol and lowest BMI
print("5 patients with highest Cholesterol:\n", df.nlargest(5, 'Cholesterol Level'))
print("5 patients with lowest BMI:\n", df.nsmallest(5, 'BMI'))


In [ ]:
original_shape = df.shape
df.drop(columns=['BMI_Category', 'Risk_Score'], inplace=True)
print(f"Shape after dropping columns: {df.shape} (was {original_shape})")

# T26: Check for duplicates, drop them, reset index
print(f"Duplicate rows count: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Shape after dropping duplicates: {df.shape}")
